In [24]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

processed_dir_2017 = Path("../data/processed/cicids17")
processed_dir_2018 = Path("../data/processed/cicids18")

processed_files_2017 = sorted(processed_dir_2017.glob("*.parquet"))
processed_files_2018 = sorted(processed_dir_2018.glob("*.parquet"))

print("2017 files:", len(processed_files_2017))
print("2018 files:", len(processed_files_2018))

2017 files: 8
2018 files: 10


In [25]:
df_2017 = pd.concat(
    [pd.read_parquet(file_path) for file_path in processed_files_2017],
    ignore_index=True
)

print("Shape:", df_2017.shape)
print()
print("Memory usage (GB):", round(df_2017.memory_usage(deep=True).sum() / 1024**3, 2))

Shape: (2830743, 71)

Memory usage (GB): 1.6


In [26]:
print("Target distribution:")
print(df_2017["Target"].value_counts())

print()
print("Target proportion:")
print(df_2017["Target"].value_counts(normalize=True).round(4))

print()
print("Original label distribution:")
print(df_2017["Original_Label"].value_counts())

Target distribution:
Target
0    2273097
1     557646
Name: count, dtype: int64

Target proportion:
Target
0    0.803
1    0.197
Name: proportion, dtype: float64

Original label distribution:
Original_Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [27]:
feature_columns = [
    column for column in df_2017.columns
    if column not in ["Original_Label", "Target", "Source_File"]
]

print("Number of modeling features:", len(feature_columns))
print("Feature schema valid:", len(feature_columns) == 68)

Number of modeling features: 68
Feature schema valid: True


In [28]:
feature_hashes = pd.util.hash_pandas_object(
    df_2017[feature_columns],
    index=False
)

duplicate_mask = feature_hashes.duplicated(keep=False)

print("Total rows:", len(df_2017))
print("Rows belonging to duplicate groups:", duplicate_mask.sum())
print("Unique duplicate fingerprints:", feature_hashes[duplicate_mask].nunique())

Total rows: 2830743
Rows belonging to duplicate groups: 688304
Unique duplicate fingerprints: 91252


In [29]:
duplicate_label_counts = (
    df_2017.loc[duplicate_mask]
    .groupby(feature_hashes[duplicate_mask])["Target"]
    .nunique()
)

conflicting_duplicate_groups = duplicate_label_counts[
    duplicate_label_counts > 1
]

print("Duplicate groups with conflicting targets:", len(conflicting_duplicate_groups))

Duplicate groups with conflicting targets: 272


In [31]:
conflicting_hash_values = set(conflicting_duplicate_groups.index)

conflicting_rows = feature_hashes.isin(conflicting_hash_values)

print("Conflicting duplicate groups:", len(conflicting_hash_values))
print("Rows belonging to conflicting groups:", conflicting_rows.sum())
print()
print(df_2017.loc[conflicting_rows, "Target"].value_counts())

Conflicting duplicate groups: 272
Rows belonging to conflicting groups: 82870

Target
1    81686
0     1184
Name: count, dtype: int64


In [32]:
conflicting_hash_values = set(conflicting_duplicate_groups.index)

conflicting_group_mask = feature_hashes.isin(conflicting_hash_values)

df_2017_retained = df_2017

df_2017_clean = df_2017.loc[
    ~conflicting_group_mask
].copy()

print("Retained dataset rows:", len(df_2017_retained))
print("Clean dataset rows:", len(df_2017_clean))
print()
print("Removed rows:", conflicting_group_mask.sum())
print("Remaining conflicting groups:",
      pd.util.hash_pandas_object(
          df_2017_clean[feature_columns],
          index=False
      ).isin(conflicting_hash_values).sum())

Retained dataset rows: 2830743
Clean dataset rows: 2747873

Removed rows: 82870
Remaining conflicting groups: 0


In [33]:
print("Retained dataset class distribution:")
print(df_2017_retained["Target"].value_counts())
print()
print(df_2017_retained["Target"].value_counts(normalize=True).round(4))

print()
print("Clean dataset class distribution:")
print(df_2017_clean["Target"].value_counts())
print()
print(df_2017_clean["Target"].value_counts(normalize=True).round(4))

Retained dataset class distribution:
Target
0    2273097
1     557646
Name: count, dtype: int64

Target
0    0.803
1    0.197
Name: proportion, dtype: float64

Clean dataset class distribution:
Target
0    2271913
1     475960
Name: count, dtype: int64

Target
0    0.8268
1    0.1732
Name: proportion, dtype: float64


In [15]:
from sklearn.model_selection import StratifiedGroupKFold

def create_stratified_group_split(df, feature_columns, random_state=42):
    groups = pd.util.hash_pandas_object(
        df[feature_columns],
        index=False
    )

    splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    train_idx, test_idx = next(
        splitter.split(
            df[feature_columns],
            df["Target"],
            groups=groups
        )
    )

    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    assert len(train_groups.intersection(test_groups)) == 0
    assert len(set(train_idx).intersection(test_idx)) == 0
    assert len(train_idx) + len(test_idx) == len(df)

    return train_idx, test_idx, groups


train_idx_retained, test_idx_retained, groups_retained = (
    create_stratified_group_split(
        df_2017_retained,
        feature_columns,
        random_state=42
    )
)

train_idx_clean, test_idx_clean, groups_clean = (
    create_stratified_group_split(
        df_2017_clean,
        feature_columns,
        random_state=42
    )
)

print("Retained dataset:")
print("Train rows:", len(train_idx_retained))
print("Test rows:", len(test_idx_retained))

print()
print("Clean dataset:")
print("Train rows:", len(train_idx_clean))
print("Test rows:", len(test_idx_clean))

Retained dataset:
Train rows: 2264594
Test rows: 566149

Clean dataset:
Train rows: 2198298
Test rows: 549575


In [16]:
from pathlib import Path
import numpy as np
import pandas as pd

split_dir = Path("../outputs/splits")
split_dir.mkdir(parents=True, exist_ok=True)

np.save(
    split_dir / "retained_train_idx.npy",
    train_idx_retained
)

np.save(
    split_dir / "retained_test_idx.npy",
    test_idx_retained
)

print("2017 split indices saved.")

2017 split indices saved.


In [17]:
from pathlib import Path
import numpy as np

split_dir = Path("../outputs/splits")
split_dir.mkdir(parents=True, exist_ok=True)

np.save(
    split_dir / "feature_columns_2017.npy",
    np.array(feature_columns, dtype=object)
)

print("2017 feature list saved.")

2017 feature list saved.


In [24]:
def verify_split(df, feature_columns, train_idx, test_idx, groups):
    total_attack_rate = df["Target"].mean()
    train_attack_rate = df.iloc[train_idx]["Target"].mean()
    test_attack_rate = df.iloc[test_idx]["Target"].mean()

    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    overlapping_groups = train_groups.intersection(test_groups)

    print("Total rows:", len(df))
    print("Train rows:", len(train_idx))
    print("Test rows:", len(test_idx))
    print()

    print("Total attack rate:", round(total_attack_rate, 4))
    print("Train attack rate:", round(train_attack_rate, 4))
    print("Test attack rate:", round(test_attack_rate, 4))
    print()

    print("Train group count:", len(train_groups))
    print("Test group count:", len(test_groups))
    print("Overlapping fingerprint groups:", len(overlapping_groups))


print("RETAINED DATASET")
verify_split(
    df_2017_retained,
    feature_columns,
    train_idx_retained,
    test_idx_retained,
    groups_retained
)

print()
print("=" * 60)
print()

print("CLEAN DATASET")
verify_split(
    df_2017_clean,
    feature_columns,
    train_idx_clean,
    test_idx_clean,
    groups_clean
)

RETAINED DATASET
Total rows: 2830743
Train rows: 2264594
Test rows: 566149

Total attack rate: 0.197
Train attack rate: 0.197
Test attack rate: 0.197

Train group count: 1786952
Test group count: 446739
Overlapping fingerprint groups: 0


CLEAN DATASET
Total rows: 2747873
Train rows: 2198298
Test rows: 549575

Total attack rate: 0.1732
Train attack rate: 0.1732
Test attack rate: 0.1732

Train group count: 1786734
Test group count: 446685
Overlapping fingerprint groups: 0


In [25]:
X_retained = df_2017_retained[feature_columns]
y_retained = df_2017_retained["Target"]

X_clean = df_2017_clean[feature_columns]
y_clean = df_2017_clean["Target"]

X_train_retained = X_retained.iloc[train_idx_retained]
X_test_retained = X_retained.iloc[test_idx_retained]

X_train_clean = X_clean.iloc[train_idx_clean]
X_test_clean = X_clean.iloc[test_idx_clean]

y_train_retained = y_retained.iloc[train_idx_retained]
y_test_retained = y_retained.iloc[test_idx_retained]

y_train_clean = y_clean.iloc[train_idx_clean]
y_test_clean = y_clean.iloc[test_idx_clean]

print("Retained:")
print("X_train:", X_train_retained.shape)
print("X_test:", X_test_retained.shape)

print()
print("Clean:")
print("X_train:", X_train_clean.shape)
print("X_test:", X_test_clean.shape)

Retained:
X_train: (2264594, 68)
X_test: (566149, 68)

Clean:
X_train: (2198298, 68)
X_test: (549575, 68)


In [26]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline_retained = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_pipeline_clean = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

print("Random Forest pipelines created.")

Random Forest pipelines created.


In [ ]:
rf_pipeline_retained.fit(
    X_train_retained,
    y_train_retained
)

print("Retained Random Forest training completed.")

rf_pipeline_clean.fit(
    X_train_clean,
    y_train_clean
)

print("Clean Random Forest training completed.")

Retained Random Forest training completed.
Clean Random Forest training completed.


In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob)
    }

    return results, confusion_matrix(y_test, y_pred)


rf_results_retained, rf_cm_retained = evaluate_model(
    rf_pipeline_retained,
    X_test_retained,
    y_test_retained
)

rf_results_clean, rf_cm_clean = evaluate_model(
    rf_pipeline_clean,
    X_test_clean,
    y_test_clean
)

print("Retained Random Forest:")
for metric, value in rf_results_retained.items():
    print(f"{metric}: {value:.4f}")

print()
print("Confusion Matrix:")
print(rf_cm_retained)

print()
print("=" * 50)
print()

print("Clean Random Forest:")
for metric, value in rf_results_clean.items():
    print(f"{metric}: {value:.4f}")

print()
print("Confusion Matrix:")
print(rf_cm_clean)

Retained Random Forest:
Accuracy: 0.9785
Precision: 0.9962
Recall: 0.8944
F1: 0.9426
ROC-AUC: 0.9996
PR-AUC: 0.9982

Confusion Matrix:
[[454239    380]
 [ 11779  99751]]


Clean Random Forest:
Accuracy: 0.9810
Precision: 0.9976
Recall: 0.8922
F1: 0.9420
ROC-AUC: 0.9996
PR-AUC: 0.9983

Confusion Matrix:
[[454176    207]
 [ 10258  84934]]


In [7]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [8]:
xgb_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

xgb_model_retained = xgb.XGBClassifier(**xgb_params)

xgb_model_clean = xgb.XGBClassifier(**xgb_params)

print("XGBoost baseline models created.")

XGBoost baseline models created.


In [ ]:
xgb_model_retained.fit(
    X_train_retained,
    y_train_retained
)

print("Retained XGBoost training completed.")

xgb_model_clean.fit(
    X_train_clean,
    y_train_clean
)

print("Clean XGBoost training completed.")

Retained XGBoost training completed.
Clean XGBoost training completed.


In [ ]:
xgb_results_retained, xgb_cm_retained = evaluate_model(
    xgb_model_retained,
    X_test_retained,
    y_test_retained
)

xgb_results_clean, xgb_cm_clean = evaluate_model(
    xgb_model_clean,
    X_test_clean,
    y_test_clean
)

print("Retained XGBoost:")
for metric, value in xgb_results_retained.items():
    print(f"{metric}: {value:.4f}")

print()
print("Confusion Matrix:")
print(xgb_cm_retained)

print()
print("=" * 50)
print()

print("Clean XGBoost:")
for metric, value in xgb_results_clean.items():
    print(f"{metric}: {value:.4f}")

print()
print("Confusion Matrix:")
print(xgb_cm_clean)

Retained XGBoost:
Accuracy: 0.9990
Precision: 0.9964
Recall: 0.9983
F1: 0.9973
ROC-AUC: 1.0000
PR-AUC: 0.9999

Confusion Matrix:
[[454219    400]
 [   194 111336]]


Clean XGBoost:
Accuracy: 0.9992
Precision: 0.9983
Recall: 0.9971
F1: 0.9977
ROC-AUC: 1.0000
PR-AUC: 1.0000

Confusion Matrix:
[[454217    166]
 [   275  94917]]


In [ ]:
from sklearn.metrics import roc_auc_score

single_feature_auc = []

for feature in feature_columns:
    train_feature = X_train_retained[feature]
    test_feature = X_test_retained[feature]

    train_valid = train_feature.notna()

    if train_valid.sum() == 0:
        continue

    try:
        train_auc = roc_auc_score(
            y_train_retained.loc[train_feature.index[train_valid]],
            train_feature.loc[train_valid]
        )

        train_auc = max(train_auc, 1 - train_auc)

        single_feature_auc.append(
            (feature, train_auc)
        )
    except ValueError:
        continue

single_feature_auc = sorted(
    single_feature_auc,
    key=lambda x: x[1],
    reverse=True
)

print("Top 20 individual features by training ROC-AUC:")
print()

for feature, auc in single_feature_auc[:20]:
    print(f"{feature}: {auc:.6f}")

Top 20 individual features by training ROC-AUC:

Packet Length Min: 0.741135
Fwd Packet Length Min: 0.739965
Bwd Packet Length Min: 0.689205
Init Fwd Win Bytes: 0.672454
Bwd Packet Length Std: 0.659076
Fwd IAT Std: 0.658534
Fwd Packet Length Mean: 0.655773
Avg Fwd Segment Size: 0.655773
Fwd Packet Length Max: 0.637387
Fwd Packets Length Total: 0.636109
Subflow Fwd Bytes: 0.636109
Idle Max: 0.628020
Idle Mean: 0.627562
Idle Min: 0.626269
Init Bwd Win Bytes: 0.609462
PSH Flag Count: 0.603181
Fwd IAT Max: 0.602818
Bwd IAT Std: 0.600531
Fwd IAT Mean: 0.598829
Fwd Header Length: 0.590798


In [ ]:
xgb_train_results_retained, xgb_train_cm_retained = evaluate_model(
    xgb_model_retained,
    X_train_retained,
    y_train_retained
)

print("Retained XGBoost - Training Performance")
for metric, value in xgb_train_results_retained.items():
    print(f"{metric}: {value:.6f}")

print()
print("Training Confusion Matrix:")
print(xgb_train_cm_retained)

print()
print("Retained XGBoost - Test Performance")
for metric, value in xgb_results_retained.items():
    print(f"{metric}: {value:.6f}")

print()
print("Test Confusion Matrix:")
print(xgb_cm_retained)

Retained XGBoost - Training Performance
Accuracy: 0.999059
Precision: 0.996809
Recall: 0.998417
F1: 0.997612
ROC-AUC: 0.999967
PR-AUC: 0.999861

Training Confusion Matrix:
[[1817052    1426]
 [    706  445410]]

Retained XGBoost - Test Performance
Accuracy: 0.998951
Precision: 0.996420
Recall: 0.998261
F1: 0.997339
ROC-AUC: 0.999967
PR-AUC: 0.999856

Test Confusion Matrix:
[[454219    400]
 [   194 111336]]


In [ ]:
test_results_retained = df_2017_retained.iloc[test_idx_retained].copy()

test_results_retained["Predicted"] = xgb_model_retained.predict(
    X_test_retained
)

test_results_retained["Probability"] = xgb_model_retained.predict_proba(
    X_test_retained
)[:, 1]

source_file_performance = []

for source_file, group in test_results_retained.groupby("Source_File"):
    y_true = group["Target"]
    y_pred = group["Predicted"]
    y_prob = group["Probability"]

    source_file_performance.append({
        "Source_File": source_file,
        "Rows": len(group),
        "Attack_Rate": y_true.mean(),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob) if y_true.nunique() > 1 else np.nan,
        "PR-AUC": average_precision_score(y_true, y_prob) if y_true.nunique() > 1 else np.nan
    })

source_file_performance = pd.DataFrame(source_file_performance)

source_file_performance.sort_values(
    "F1",
    ascending=False
)

,Source_File,Rows,Attack_Rate,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,55137,0.534995,0.999837,0.999729,0.999966,0.999847,1.000000,1.000000
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,45455,0.568232,0.999626,0.999884,0.999458,0.999671,0.999999,0.999999
7,Wednesday-workingHours.pcap_ISCX.csv,140754,0.373346,0.999524,0.999068,0.999657,0.999363,0.999992,0.999986
6,Tuesday-WorkingHours.pcap_ISCX.csv,88244,0.031356,0.999569,0.991004,0.995302,0.993148,0.999993,0.999786
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,34616,0.012740,0.998989,0.990338,0.929705,0.959064,0.999942,0.995823
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,38801,0.011340,0.996675,0.961424,0.736364,0.833977,0.999692,0.965122
4,Thursday-WorkingHours-Afternoon-Infilteration....,54880,0.000091,0.995262,0.015209,0.800000,0.029851,0.933762,0.212596
3,Monday-WorkingHours.pcap_ISCX.csv,108262,0.000000,0.999640,0.000000,0.000000,0.000000,NaN,NaN


In [ ]:
test_label_performance = []

for label, group in test_results_retained.groupby("Original_Label"):
    y_true = group["Target"]
    y_pred = group["Predicted"]
    y_prob = group["Probability"]

    test_label_performance.append({
        "Original_Label": label,
        "Rows": len(group),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob) if y_true.nunique() > 1 else np.nan,
        "PR-AUC": average_precision_score(y_true, y_prob) if y_true.nunique() > 1 else np.nan
    })

test_label_performance = pd.DataFrame(test_label_performance)

test_label_performance.sort_values(
    "Rows",
    ascending=False
)

NameError: name 'test_results_retained' is not defined

In [15]:
scenario_summary = (
    df_2017_retained
    .groupby("Source_File")
    .agg(
        Rows=("Target", "size"),
        Attack_Rate=("Target", "mean"),
        Attack_Rows=("Target", "sum"),
        Benign_Rows=("Target", lambda x: (x == 0).sum())
    )
    .sort_values("Rows", ascending=False)
)

scenario_summary

,Rows,Attack_Rate,Attack_Rows,Benign_Rows
Source_File,,,,
Wednesday-workingHours.pcap_ISCX.csv,692703,0.364762,252672,440031
Monday-WorkingHours.pcap_ISCX.csv,529918,0.000000,0,529918
Tuesday-WorkingHours.pcap_ISCX.csv,445909,0.031027,13835,432074
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,288602,0.000125,36,288566
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,286467,0.554793,158930,127537
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0.567131,128027,97718
Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0.010291,1966,189067
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,170366,0.012796,2180,168186


In [17]:
scenario_label_overlap = []

for test_file in scenario_files:
    train_labels = set(
        df_2017_retained.loc[
            df_2017_retained["Source_File"] != test_file,
            "Original_Label"
        ].unique()
    )

    test_labels = set(
        df_2017_retained.loc[
            df_2017_retained["Source_File"] == test_file,
            "Original_Label"
        ].unique()
    )

    unseen_labels = test_labels - train_labels

    scenario_label_overlap.append({
        "Test_File": test_file,
        "Test_Labels": sorted(test_labels),
        "Unseen_in_Train": sorted(unseen_labels)
    })

scenario_label_overlap = pd.DataFrame(scenario_label_overlap)

scenario_label_overlap

,Test_File,Test_Labels,Unseen_in_Train
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,"[BENIGN, DDoS]",[DDoS]
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,"[BENIGN, PortScan]",[PortScan]
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,"[BENIGN, Bot]",[Bot]
3,Monday-WorkingHours.pcap_ISCX.csv,[BENIGN],[]
4,Thursday-WorkingHours-Afternoon-Infilteration....,"[BENIGN, Infiltration]",[Infiltration]
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,"[BENIGN, Web Attack � Brute Force, Web Attack ...","[Web Attack � Brute Force, Web Attack � Sql In..."
6,Tuesday-WorkingHours.pcap_ISCX.csv,"[BENIGN, FTP-Patator, SSH-Patator]","[FTP-Patator, SSH-Patator]"
7,Wednesday-workingHours.pcap_ISCX.csv,"[BENIGN, DoS GoldenEye, DoS Hulk, DoS Slowhttp...","[DoS GoldenEye, DoS Hulk, DoS Slowhttptest, Do..."


In [20]:
import gc
import time
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

scenario_results = []
scenario_confusion_matrices = {}

X = df_2017_retained[feature_columns]
y = df_2017_retained["Target"]
source_files = df_2017_retained["Source_File"]

unique_files = sorted(source_files.unique())

for held_out_file in unique_files:
    train_mask = source_files != held_out_file
    test_mask = source_files == held_out_file

    X_train_scenario = X.loc[train_mask]
    X_test_scenario = X.loc[test_mask]
    y_train_scenario = y.loc[train_mask]
    y_test_scenario = y.loc[test_mask]

    model = xgb.XGBClassifier(**xgb_params)

    start_time = time.time()

    model.fit(
        X_train_scenario,
        y_train_scenario
    )

    training_time = time.time() - start_time

    y_pred = model.predict(X_test_scenario)
    y_prob = model.predict_proba(X_test_scenario)[:, 1]

    has_attacks = y_test_scenario.sum() > 0

    if has_attacks:
        precision = precision_score(
            y_test_scenario,
            y_pred,
            zero_division=0
        )
        recall = recall_score(
            y_test_scenario,
            y_pred,
            zero_division=0
        )
        f1 = f1_score(
            y_test_scenario,
            y_pred,
            zero_division=0
        )
        roc_auc = roc_auc_score(
            y_test_scenario,
            y_prob
        )
        pr_auc = average_precision_score(
            y_test_scenario,
            y_prob
        )
    else:
        precision = np.nan
        recall = np.nan
        f1 = np.nan
        roc_auc = np.nan
        pr_auc = np.nan

    results = {
        "Held_Out_File": held_out_file,
        "Train_Rows": len(y_train_scenario),
        "Test_Rows": len(y_test_scenario),
        "Train_Attack_Rate": y_train_scenario.mean(),
        "Test_Attack_Rate": y_test_scenario.mean(),
        "Test_Attack_Rows": int(y_test_scenario.sum()),
        "Accuracy": accuracy_score(y_test_scenario, y_pred),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Training_Time_Minutes": training_time / 60
    }

    scenario_results.append(results)
    scenario_confusion_matrices[held_out_file] = confusion_matrix(
        y_test_scenario,
        y_pred
    )

    print(f"\nCompleted: {held_out_file}")
    print(f"Training time: {training_time / 60:.2f} minutes")
    print(f"Test rows: {len(y_test_scenario):,}")
    print(f"Test attacks: {int(y_test_scenario.sum()):,}")
    print(f"Accuracy: {results['Accuracy']:.6f}")

    if has_attacks:
        print(f"Precision: {precision:.6f}")
        print(f"Recall: {recall:.6f}")
        print(f"F1: {f1:.6f}")
        print(f"ROC-AUC: {roc_auc:.6f}")
        print(f"PR-AUC: {pr_auc:.6f}")

    del X_train_scenario
    del X_test_scenario
    del y_train_scenario
    del y_test_scenario
    del model
    del y_pred
    del y_prob

    gc.collect()

scenario_results_df = pd.DataFrame(scenario_results)

scenario_results_df


Completed: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Training time: 1.37 minutes
Test rows: 225,745
Test attacks: 128,027
Accuracy: 0.667279
Precision: 0.987759
Recall: 0.418513
F1: 0.587924
ROC-AUC: 0.863836
PR-AUC: 0.900806

Completed: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Training time: 1.65 minutes
Test rows: 286,467
Test attacks: 158,930
Accuracy: 0.444177
Precision: 0.356378
Recall: 0.002303
F1: 0.004576
ROC-AUC: 0.919503
PR-AUC: 0.878225

Completed: Friday-WorkingHours-Morning.pcap_ISCX.csv
Training time: 2.31 minutes
Test rows: 191,033
Test attacks: 1,966
Accuracy: 0.989384
Precision: 0.000000
Recall: 0.000000
F1: 0.000000
ROC-AUC: 0.922291
PR-AUC: 0.109889

Completed: Monday-WorkingHours.pcap_ISCX.csv
Training time: 1.95 minutes
Test rows: 529,918
Test attacks: 0
Accuracy: 0.999353

Completed: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Training time: 1.22 minutes
Test rows: 288,602
Test attacks: 36
Accuracy: 0.780386
Precision: 0.000016


,Held_Out_File,Train_Rows,Test_Rows,Train_Attack_Rate,Test_Attack_Rate,Test_Attack_Rows,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Training_Time_Minutes
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,2604998,225745,0.164921,0.567131,128027,0.667279,0.987759,0.418513,0.587924,0.863836,0.900806,1.373363
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,2544276,286467,0.156711,0.554793,158930,0.444177,0.356378,0.002303,0.004576,0.919503,0.878225,1.648714
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,2639710,191033,0.210508,0.010291,1966,0.989384,0.000000,0.000000,0.000000,0.922291,0.109889,2.310542
3,Monday-WorkingHours.pcap_ISCX.csv,2300825,529918,0.242368,0.000000,0,0.999353,NaN,NaN,NaN,NaN,NaN,1.946718
4,Thursday-WorkingHours-Afternoon-Infilteration....,2542141,288602,0.219347,0.000125,36,0.780386,0.000016,0.027778,0.000032,0.616092,0.000152,1.221047
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,2660377,170366,0.208792,0.012796,2180,0.997018,0.959846,0.800459,0.872936,0.996078,0.859932,1.282250
6,Tuesday-WorkingHours.pcap_ISCX.csv,2384834,445909,0.228029,0.031027,13835,0.966948,0.021209,0.001446,0.002707,0.902396,0.155945,1.094003
7,Wednesday-workingHours.pcap_ISCX.csv,2138040,692703,0.142642,0.364762,252672,0.656914,0.993363,0.059825,0.112853,0.964544,0.935798,1.012299


In [21]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from xgboost import XGBClassifier

tuning_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "scale_pos_weight": [1.0, 2.0, 4.0]
}

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=tuning_model,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="average_precision",
    cv=cv_strategy,
    verbose=2,
    random_state=42,
    n_jobs=1,
    return_train_score=True
)

print("Tuning setup ready.")
print("CV folds:", cv_strategy.n_splits)
print("Random configurations:", random_search.n_iter)
print("Scoring metric: PR-AUC")

Tuning setup ready.
CV folds: 3
Random configurations: 12
Scoring metric: PR-AUC


In [27]:
start_time = time.time()

random_search.fit(
    X_train_retained,
    y_train_retained
)

tuning_time = time.time() - start_time

print("Tuning completed.")
print(f"Total tuning time: {tuning_time / 60:.2f} minutes")
print("\nBest parameters:")
print(random_search.best_params_)
print("\nBest CV PR-AUC:")
print(random_search.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, min_child_weight=10, n_estimators=100, scale_pos_weight=4.0, subsample=1.0; total time=  22.4s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, min_child_weight=10, n_estimators=100, scale_pos_weight=4.0, subsample=1.0; total time=  22.8s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, min_child_weight=10, n_estimators=100, scale_pos_weight=4.0, subsample=1.0; total time=  25.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, min_child_weight=1, n_estimators=100, scale_pos_weight=4.0, subsample=0.8; total time=  28.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, min_child_weight=1, n_estimators=100, scale_pos_weight=4.0, subsample=0.8; total time=  39.5s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, min_child_weight=1, n_estimators=100, scale_pos_weight=4.0, subsample=0.8; total time=  

In [9]:
best_params = random_search.best_params_

xgb_tuned_2017 = xgb.XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

xgb_tuned_2017.fit(
    X_train_retained,
    y_train_retained
)

final_training_time = time.time() - start_time

print("Final tuned model trained.")
print(f"Training time: {final_training_time / 60:.2f} minutes")

NameError: name 'random_search' is not defined

In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob)
    }

    return results, confusion_matrix(y_test, y_pred)

In [31]:
tuned_results_2017, tuned_cm_2017 = evaluate_model(
    xgb_tuned_2017,
    X_test_retained,
    y_test_retained
)

print("Tuned 2017 Test Results:")
for metric, value in tuned_results_2017.items():
    print(f"{metric}: {value:.6f}")

print("\nConfusion Matrix:")
print(tuned_cm_2017)

Tuned 2017 Test Results:
Accuracy: 0.999029
Precision: 0.996590
Recall: 0.998485
F1: 0.997537
ROC-AUC: 0.999973
PR-AUC: 0.999878

Confusion Matrix:
[[454238    381]
 [   169 111361]]


Testing on 2018 training on 2017

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

processed_dir_2018 = Path("../data/processed/cicids18")

processed_files_2018 = sorted(
    processed_dir_2018.glob("*.parquet")
)

print("CSE-CIC-IDS2018 files:", len(processed_files_2018))
for file_path in processed_files_2018:
    print(file_path.name)

CSE-CIC-IDS2018 files: 10
Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet


In [11]:
df_2018 = pd.concat(
    [pd.read_parquet(file_path) for file_path in processed_files_2018],
    ignore_index=True
)

print("2018 shape:", df_2018.shape)
print("\nColumns:", len(df_2018.columns))
print("\nTarget distribution:")
print(df_2018["Target"].value_counts())

2018 shape: (6659532, 71)

Columns: 71

Target distribution:
Target
0    5329008
1    1330524
Name: count, dtype: int64


In [12]:
feature_columns_2018 = [
    column for column in df_2018.columns
    if column not in ["Original_Label", "Target", "Source_File"]
]

print("2017 feature count:", len(feature_columns))
print("2018 feature count:", len(feature_columns_2018))

missing_in_2018 = sorted(
    set(feature_columns) - set(feature_columns_2018)
)

extra_in_2018 = sorted(
    set(feature_columns_2018) - set(feature_columns)
)

print("\nMissing from 2018:", missing_in_2018)
print("Extra in 2018:", extra_in_2018)

assert len(feature_columns) == 68
assert len(feature_columns_2018) == 68
assert set(feature_columns) == set(feature_columns_2018)

print("\nSchema verification PASSED.")

2017 feature count: 68
2018 feature count: 68

Missing from 2018: []
Extra in 2018: []

Schema verification PASSED.


In [35]:
X_2018 = df_2018[feature_columns]
y_2018 = df_2018["Target"]

print("X_2018 shape:", X_2018.shape)
print("y_2018 shape:", y_2018.shape)

print("\n2018 class distribution:")
print(y_2018.value_counts())

print("\n2018 class proportions:")
print(y_2018.value_counts(normalize=True))

X_2018 shape: (6659532, 68)
y_2018 shape: (6659532,)

2018 class distribution:
Target
0    5329008
1    1330524
Name: count, dtype: int64

2018 class proportions:
Target
0    0.800208
1    0.199792
Name: proportion, dtype: float64


In [36]:
print("Missing values:", X_2018.isna().sum().sum())
print("Infinite values:", np.isinf(X_2018.select_dtypes(include=np.number)).sum().sum())
print("Target values:", sorted(y_2018.unique()))

assert X_2018.shape[1] == 68
assert X_2018.shape[0] == len(y_2018)
assert y_2018.isin([0, 1]).all()
assert np.isinf(X_2018.select_dtypes(include=np.number)).sum().sum() == 0

print("\n2018 external test data verification PASSED.")

Missing values: 0
Infinite values: 0
Target values: [np.int8(0), np.int8(1)]

2018 external test data verification PASSED.


In [37]:
cross_results_2017_to_2018, cross_cm_2017_to_2018 = evaluate_model(
    xgb_tuned_2017,
    X_2018,
    y_2018
)

print("2017 → 2018 Cross-Dataset Results:")

for metric, value in cross_results_2017_to_2018.items():
    print(f"{metric}: {value:.6f}")

print("\nConfusion Matrix:")
print(cross_cm_2017_to_2018)

2017 → 2018 Cross-Dataset Results:
Accuracy: 0.801322
Precision: 0.866640
Recall: 0.006594
F1: 0.013088
ROC-AUC: 0.778893
PR-AUC: 0.574965

Confusion Matrix:
[[5327658    1350]
 [1321751    8773]]


In [38]:
from scipy.stats import ks_2samp

sample_size = 100000
random_state = 42

X_2017_shift = X_train_retained.sample(
    n=min(sample_size, len(X_train_retained)),
    random_state=random_state
)

X_2018_shift = X_2018.sample(
    n=min(sample_size, len(X_2018)),
    random_state=random_state
)

shift_results = []

for feature in feature_columns:
    values_2017 = X_2017_shift[feature].dropna()
    values_2018 = X_2018_shift[feature].dropna()

    ks_stat, p_value = ks_2samp(
        values_2017,
        values_2018
    )

    median_2017 = values_2017.median()
    median_2018 = values_2018.median()

    mean_2017 = values_2017.mean()
    mean_2018 = values_2018.mean()

    shift_results.append({
        "Feature": feature,
        "KS_Statistic": ks_stat,
        "P_Value": p_value,
        "Mean_2017": mean_2017,
        "Mean_2018": mean_2018,
        "Median_2017": median_2017,
        "Median_2018": median_2018
    })

shift_df = pd.DataFrame(shift_results)

shift_df = shift_df.sort_values(
    "KS_Statistic",
    ascending=False
).reset_index(drop=True)

shift_df.head(20)

,Feature,KS_Statistic,P_Value,Mean_2017,Mean_2018,Median_2017,Median_2018
0,Fwd IAT Total,0.398160,0.0,1.440096e+07,1.510660e+07,47.000000,109932.500000
1,Fwd IAT Min,0.394530,0.0,1.050058e+06,5.382551e+06,3.000000,64.000000
2,Fwd Seg Size Min,0.394370,0.0,2.611440e+01,1.864808e+01,24.000000,20.000000
3,Fwd IAT Max,0.392980,0.0,8.970003e+06,8.627320e+06,47.000000,75271.000000
4,Fwd IAT Mean,0.388910,0.0,2.620821e+06,6.130815e+06,45.000000,39503.000000
5,Fwd Packet Length Min,0.329980,0.0,1.889403e+01,9.038940e+00,2.000000,0.000000
6,Fwd Packets/s,0.327070,0.0,6.110713e+04,4.594716e+03,62.613487,6.815581
7,Flow Packets/s,0.324838,0.0,6.805606e+04,5.898386e+03,116.290660,11.898592
8,Packet Length Min,0.320900,0.0,1.643846e+01,8.926270e+00,2.000000,0.000000
9,Flow Duration,0.318980,0.0,1.470649e+07,1.552425e+07,31286.000000,672368.500000


In [39]:
print("Top 20 features by distribution shift:")

display(
    shift_df[
        [
            "Feature",
            "KS_Statistic",
            "P_Value",
            "Mean_2017",
            "Mean_2018",
            "Median_2017",
            "Median_2018"
        ]
    ].head(20)
)

Top 20 features by distribution shift:


,Feature,KS_Statistic,P_Value,Mean_2017,Mean_2018,Median_2017,Median_2018
0,Fwd IAT Total,0.398160,0.0,1.440096e+07,1.510660e+07,47.000000,109932.500000
1,Fwd IAT Min,0.394530,0.0,1.050058e+06,5.382551e+06,3.000000,64.000000
2,Fwd Seg Size Min,0.394370,0.0,2.611440e+01,1.864808e+01,24.000000,20.000000
3,Fwd IAT Max,0.392980,0.0,8.970003e+06,8.627320e+06,47.000000,75271.000000
4,Fwd IAT Mean,0.388910,0.0,2.620821e+06,6.130815e+06,45.000000,39503.000000
5,Fwd Packet Length Min,0.329980,0.0,1.889403e+01,9.038940e+00,2.000000,0.000000
6,Fwd Packets/s,0.327070,0.0,6.110713e+04,4.594716e+03,62.613487,6.815581
7,Flow Packets/s,0.324838,0.0,6.805606e+04,5.898386e+03,116.290660,11.898592
8,Packet Length Min,0.320900,0.0,1.643846e+01,8.926270e+00,2.000000,0.000000
9,Flow Duration,0.318980,0.0,1.470649e+07,1.552425e+07,31286.000000,672368.500000


In [40]:
print("Mean KS statistic:", shift_df["KS_Statistic"].mean())
print("Median KS statistic:", shift_df["KS_Statistic"].median())

print("\nFeatures with KS > 0.10:")
print((shift_df["KS_Statistic"] > 0.10).sum())

print("\nFeatures with KS > 0.20:")
print((shift_df["KS_Statistic"] > 0.20).sum())

print("\nFeatures with KS > 0.30:")
print((shift_df["KS_Statistic"] > 0.30).sum())

Mean KS statistic: 0.20517053439023197
Median KS statistic: 0.23367999999999997

Features with KS > 0.10:
52

Features with KS > 0.20:
42

Features with KS > 0.30:
12


In [41]:
y_pred_2018 = xgb_tuned_2017.predict(X_2018)
y_prob_2018 = xgb_tuned_2017.predict_proba(X_2018)[:, 1]

df_2018_eval = df_2018[
    ["Original_Label", "Target", "Source_File"]
].copy()

df_2018_eval["Prediction"] = y_pred_2018
df_2018_eval["Probability"] = y_prob_2018

source_results_2018 = []

for source_file, group in df_2018_eval.groupby("Source_File"):
    y_true = group["Target"]
    y_pred = group["Prediction"]
    y_prob = group["Probability"]

    result = {
        "Source_File": source_file,
        "Rows": len(group),
        "Attack_Rate": y_true.mean(),
        "Attack_Rows": int(y_true.sum()),
        "Predicted_Attacks": int(y_pred.sum()),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

    if y_true.nunique() == 2:
        result["ROC-AUC"] = roc_auc_score(y_true, y_prob)
        result["PR-AUC"] = average_precision_score(y_true, y_prob)
    else:
        result["ROC-AUC"] = np.nan
        result["PR-AUC"] = np.nan

    source_results_2018.append(result)

source_results_2018 = pd.DataFrame(source_results_2018)

source_results_2018 = source_results_2018.sort_values(
    "Attack_Rate",
    ascending=False
).reset_index(drop=True)

source_results_2018

,Source_File,Rows,Attack_Rate,Attack_Rows,Predicted_Attacks,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,954846,0.602573,575364,46,0.397379,0.000000,0.000000,0.000000,0.965838,0.968670
1,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,561396,0.357307,200591,0,0.642693,0.000000,0.000000,0.000000,0.764480,0.717386
2,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,249170,0.248963,62034,91,0.750736,0.087912,0.000129,0.000258,0.512389,0.250192
3,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,591873,0.245414,145254,0,0.754586,0.000000,0.000000,0.000000,0.979520,0.938167
4,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,771587,0.187322,144535,172,0.812455,0.000000,0.000000,0.000000,0.765985,0.313369
5,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,619346,0.151936,94101,113,0.847881,0.000000,0.000000,0.000000,0.888758,0.404256
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,456873,0.123555,56449,180,0.876082,0.038889,0.000124,0.000247,0.500864,0.121761
7,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,794812,0.064561,51314,8967,0.945973,0.966879,0.168960,0.287653,0.906989,0.496762
8,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,829405,0.000652,541,332,0.999085,0.171687,0.105360,0.130584,0.826171,0.033118
9,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,830224,0.000411,341,222,0.999397,0.139640,0.090909,0.110124,0.823952,0.015133


In [42]:
print(
    source_results_2018.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

                                                      Source_File   Rows  Attack_Rate  Attack_Rows  Predicted_Attacks  Accuracy  Precision   Recall       F1  ROC-AUC   PR-AUC
       DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet 954846     0.602573       575364                 46  0.397379   0.000000 0.000000 0.000000 0.965838 0.968670
     DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet 561396     0.357307       200591                  0  0.642693   0.000000 0.000000 0.000000 0.764480 0.717386
     Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet 249170     0.248963        62034                 91  0.750736   0.087912 0.000129 0.000258 0.512389 0.250192
         DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet 591873     0.245414       145254                  0  0.754586   0.000000 0.000000 0.000000 0.979520 0.938167
       Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet 771587     0.187322       144535                172  0.8124

In [43]:
thresholds = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

threshold_results_2018 = []

for threshold in thresholds:
    y_pred_threshold = (
        y_prob_2018 >= threshold
    ).astype(np.int8)

    threshold_results_2018.append({
        "Threshold": threshold,
        "Predicted_Attacks": int(y_pred_threshold.sum()),
        "Accuracy": accuracy_score(y_2018, y_pred_threshold),
        "Precision": precision_score(
            y_2018,
            y_pred_threshold,
            zero_division=0
        ),
        "Recall": recall_score(
            y_2018,
            y_pred_threshold,
            zero_division=0
        ),
        "F1": f1_score(
            y_2018,
            y_pred_threshold,
            zero_division=0
        )
    })

threshold_results_2018 = pd.DataFrame(
    threshold_results_2018
)

threshold_results_2018

,Threshold,Predicted_Attacks,Accuracy,Precision,Recall,F1
0,0.01,306260,0.823423,0.752410,0.173190,0.281568
1,0.05,106249,0.813490,0.916253,0.073167,0.135513
2,0.10,30062,0.803373,0.850575,0.019218,0.037587
3,0.20,12662,0.801463,0.830122,0.007900,0.015651
4,0.30,12257,0.801486,0.847189,0.007804,0.015466
5,0.40,11969,0.801485,0.855293,0.007694,0.015251
6,0.50,10123,0.801322,0.866640,0.006594,0.013088


In [44]:
probability_summary_2018 = pd.DataFrame({
    "Class": ["Benign", "Attack"],
    "Mean_Probability": [
        y_prob_2018[y_2018 == 0].mean(),
        y_prob_2018[y_2018 == 1].mean()
    ],
    "Median_Probability": [
        np.median(y_prob_2018[y_2018 == 0]),
        np.median(y_prob_2018[y_2018 == 1])
    ],
    "90th_Percentile": [
        np.percentile(y_prob_2018[y_2018 == 0], 90),
        np.percentile(y_prob_2018[y_2018 == 1], 90)
    ],
    "99th_Percentile": [
        np.percentile(y_prob_2018[y_2018 == 0], 99),
        np.percentile(y_prob_2018[y_2018 == 1], 99)
    ]
})

probability_summary_2018

,Class,Mean_Probability,Median_Probability,90th_Percentile,99th_Percentile
0,Benign,0.000649,0.000005,0.000243,0.012360
1,Attack,0.013594,0.000310,0.020109,0.144164


Attack Families cross-dataset Transfer

In [13]:
import xgboost as xgb
from pathlib import Path

model_path = Path("../outputs/models/xgb_tuned_2017.json")

xgb_tuned_2017 = xgb.XGBClassifier()
xgb_tuned_2017.load_model(model_path)

print("Model loaded successfully.")

Model loaded successfully.


In [14]:
feature_columns_2018 = [
    column for column in df_2018.columns
    if column not in ["Original_Label", "Target", "Source_File"]
]

feature_hashes_2018 = pd.util.hash_pandas_object(
    df_2018[feature_columns_2018].astype(np.float64),
    index=False
)

hash_target_counts_2018 = (
    pd.DataFrame({
        "Hash": feature_hashes_2018,
        "Target": df_2018["Target"]
    })
    .groupby("Hash")["Target"]
    .nunique()
)

conflicting_hash_values_2018 = hash_target_counts_2018[
    hash_target_counts_2018 > 1
].index

conflicting_group_mask_2018 = feature_hashes_2018.isin(
    conflicting_hash_values_2018
)

df_2018_clean = df_2018.loc[
    ~conflicting_group_mask_2018
].copy()

print("Original rows:", len(df_2018))
print("Conflicting groups:", len(conflicting_hash_values_2018))
print("Removed rows:", conflicting_group_mask_2018.sum())
print("Evaluation rows:", len(df_2018_clean))

Original rows: 6659532
Conflicting groups: 21947
Removed rows: 70102
Evaluation rows: 6589430


In [16]:
print(df_2018["Original_Label"].value_counts())

Original_Label
Benign                      5329008
DDoS attacks-LOIC-HTTP       575364
DDOS attack-HOIC             198861
DoS attacks-Hulk             145199
Bot                          144535
Infilteration                118483
SSH-Bruteforce                94048
DoS attacks-GoldenEye         41406
DoS attacks-Slowloris          9908
DDOS attack-LOIC-UDP           1730
Brute Force -Web                568
Brute Force -XSS                229
SQL Injection                    85
DoS attacks-SlowHTTPTest         55
FTP-BruteForce                   53
Name: count, dtype: int64


In [17]:
print(df_2017["Original_Label"].value_counts())

Original_Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [18]:
attack_family_mapping_2018 = {
    "Benign": "BENIGN",
    "DDoS attacks-LOIC-HTTP": "DDoS",
    "DDOS attack-HOIC": "DDoS",
    "DDOS attack-LOIC-UDP": "DDoS",
    "DoS attacks-Hulk": "DoS",
    "DoS attacks-GoldenEye": "DoS",
    "DoS attacks-Slowloris": "DoS",
    "DoS attacks-SlowHTTPTest": "DoS",
    "SSH-Bruteforce": "Brute Force",
    "FTP-BruteForce": "Brute Force",
    "Brute Force -Web": "Web Attack",
    "Brute Force -XSS": "Web Attack",
    "SQL Injection": "Web Attack",
    "Bot": "Bot",
    "Infilteration": "Infiltration"
}

attack_family_2018 = df_2018["Original_Label"].map(
    attack_family_mapping_2018
)

print("Unmapped labels:")
print(
    df_2018.loc[
        attack_family_2018.isna(),
        "Original_Label"
    ].value_counts()
)

print("\nAttack-family distribution:")
print(attack_family_2018.value_counts())

Unmapped labels:
Series([], Name: count, dtype: int64)

Attack-family distribution:
Original_Label
BENIGN          5329008
DDoS             775955
DoS              196568
Bot              144535
Infiltration     118483
Brute Force       94101
Web Attack          882
Name: count, dtype: int64


In [19]:
n_rows_2018 = len(df_2018)

y_prob_2017_on_2018 = np.empty(
    n_rows_2018,
    dtype=np.float32
)

chunk_size = 200000

for start in range(0, n_rows_2018, chunk_size):
    end = min(start + chunk_size, n_rows_2018)

    X_chunk = df_2018.iloc[
        start:end
    ][feature_columns_2017].to_numpy(
        dtype=np.float32
    )

    y_prob_2017_on_2018[start:end] = (
        xgb_tuned_2017.predict_proba(X_chunk)[:, 1]
    )

print("Predictions generated:", len(y_prob_2017_on_2018))
print("Minimum probability:", y_prob_2017_on_2018.min())
print("Maximum probability:", y_prob_2017_on_2018.max())
print("Mean probability:", y_prob_2017_on_2018.mean())

Predictions generated: 6659532
Minimum probability: 4.4451234e-08
Maximum probability: 0.99902964
Mean probability: 0.0032351671


In [22]:
attack_families_2018 = [
    "DDoS",
    "DoS",
    "Brute Force",
    "Web Attack",
    "Bot",
    "Infiltration"
]

family_results_2017_to_2018 = []

benign_mask_2018 = attack_family_2018 == "BENIGN"

print("Benign rows available:", benign_mask_2018.sum())

for family in attack_families_2018:
    family_mask_2018 = attack_family_2018 == family
    evaluation_mask = benign_mask_2018 | family_mask_2018

    y_true_family = family_mask_2018[evaluation_mask].astype(np.int8)
    y_prob_family = y_prob_2017_on_2018[evaluation_mask]

    y_pred_family = (y_prob_family >= 0.5).astype(np.int8)

    family_results_2017_to_2018.append({
        "Family": family,
        "Attack_Samples": int(family_mask_2018.sum()),
        "Benign_Samples": int(benign_mask_2018.sum()),
        "Accuracy": accuracy_score(y_true_family, y_pred_family),
        "Precision": precision_score(
            y_true_family,
            y_pred_family,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true_family,
            y_pred_family,
            zero_division=0
        ),
        "F1": f1_score(
            y_true_family,
            y_pred_family,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true_family,
            y_prob_family
        ),
        "PR-AUC": average_precision_score(
            y_true_family,
            y_prob_family
        ),
        "Predicted_Attacks": int(y_pred_family.sum())
    })

family_results_2017_to_2018 = pd.DataFrame(
    family_results_2017_to_2018
).reset_index(drop=True)

print(family_results_2017_to_2018.to_string(index=False))

Benign rows available: 5329008
      Family  Attack_Samples  Benign_Samples  Accuracy  Precision   Recall       F1  ROC-AUC   PR-AUC  Predicted_Attacks
        DDoS          775955         5329008  0.872677   0.000000 0.000000 0.000000 0.776025 0.576451               1350
         DoS          196568         5329008  0.965751   0.865269 0.044107 0.083935 0.902411 0.228240              10020
 Brute Force           94101         5329008  0.982399   0.000000 0.000000 0.000000 0.906065 0.081233               1350
  Web Attack             882         5329008  0.999598   0.061196 0.099773 0.075862 0.871440 0.009366               1438
         Bot          144535         5329008  0.973347   0.000000 0.000000 0.000000 0.783545 0.058155               1350
Infiltration          118483         5329008  0.978005   0.010989 0.000127 0.000250 0.485391 0.021304               1365


In [23]:
from pathlib import Path

table_dir = Path("../outputs/tables")
table_dir.mkdir(parents=True, exist_ok=True)

family_results_2017_to_2018.to_csv(
    table_dir / "attack_family_transfer_2017_to_2018.csv",
    index=False
)

print(
    "Saved:",
    table_dir / "attack_family_transfer_2017_to_2018.csv"
)

Saved: ..\outputs\tables\attack_family_transfer_2017_to_2018.csv


In [34]:
feature_cols = feature_columns_2017

def missing_audit(df, name):
    X = df[feature_cols]
    y = df["Target"]

    overall = X.isna().sum()
    benign = X[y == 0].isna().sum()
    attack = X[y == 1].isna().sum()

    result = pd.DataFrame({
        "Feature": feature_cols,
        "Missing_Total": overall.values,
        "Missing_Benign": benign.values,
        "Missing_Attack": attack.values,
        "Missing_Total_%": (overall.values / len(df) * 100),
        "Missing_Benign_%": (benign.values / (y == 0).sum() * 100),
        "Missing_Attack_%": (attack.values / (y == 1).sum() * 100)
    })

    result = result.sort_values("Missing_Total", ascending=False).reset_index(drop=True)

    rows_any = X.isna().any(axis=1)

    summary = pd.DataFrame({
        "Dataset": [name],
        "Rows": [len(df)],
        "Rows_With_Any_Missing": [rows_any.sum()],
        "Rows_With_Any_Missing_%": [rows_any.mean() * 100],
        "Benign_Rows_With_Missing": [rows_any[y == 0].sum()],
        "Attack_Rows_With_Missing": [rows_any[y == 1].sum()],
        "Total_Missing_Cells": [X.isna().sum().sum()]
    })

    file_audit = (
        df.assign(_has_missing=rows_any)
        .groupby("Source_File")["_has_missing"]
        .agg(["count", "sum"])
        .rename(columns={"count": "Rows", "sum": "Rows_With_Missing"})
        .reset_index()
    )

    file_audit["Rows_With_Missing_%"] = (
        file_audit["Rows_With_Missing"] / file_audit["Rows"] * 100
    )

    return result, summary, file_audit


audit_retained, summary_retained, file_retained = missing_audit(
    df_2017_retained,
    "2017 Retained"
)

audit_clean, summary_clean, file_clean = missing_audit(
    df_2017_clean,
    "2017 Clean"
)

print("RETAINED SUMMARY")
print(summary_retained.to_string(index=False))

print("\nCLEAN SUMMARY")
print(summary_clean.to_string(index=False))

print("\nTOP MISSING FEATURES — RETAINED")
print(audit_retained.head(20).to_string(index=False))

print("\nTOP MISSING FEATURES — CLEAN")
print(audit_clean.head(20).to_string(index=False))

print("\nMISSINGNESS BY FILE — RETAINED")
print(file_retained.to_string(index=False))


RETAINED SUMMARY
      Dataset    Rows  Rows_With_Any_Missing  Rows_With_Any_Missing_%  Benign_Rows_With_Missing  Attack_Rows_With_Missing  Total_Missing_Cells
2017 Retained 2830743                   2867                 0.101281                      1777                      1090                 5734

CLEAN SUMMARY
   Dataset    Rows  Rows_With_Any_Missing  Rows_With_Any_Missing_%  Benign_Rows_With_Missing  Attack_Rows_With_Missing  Total_Missing_Cells
2017 Clean 2747873                   2455                 0.089342                      1773                       682                 4910

TOP MISSING FEATURES — RETAINED
                 Feature  Missing_Total  Missing_Benign  Missing_Attack  Missing_Total_%  Missing_Benign_%  Missing_Attack_%
            Flow Bytes/s           2867            1777            1090         0.101281          0.078175          0.195465
          Flow Packets/s           2867            1777            1090         0.101281          0.078175          0.1

In [37]:
from pathlib import Path
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

split_dir = Path("../outputs/splits")

candidate_paths = [
    split_dir / "retained_test_idx.npy",
    split_dir / "retained_train_idx.npy",
]

test_idx_path = next((p for p in candidate_paths if p.exists()), None)

if test_idx_path is None:
    matches = sorted(split_dir.glob("*2017*test*.npy"))
    if not matches:
        matches = sorted(split_dir.glob("*test*2017*.npy"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Could not identify the 2017 test-index file. Matches: {matches}")
    test_idx_path = matches[0]

test_idx_2017 = np.load(test_idx_path)

df_test_2017 = df_2017_retained.iloc[test_idx_2017].copy()

X_test_saved = df_test_2017[feature_cols]
y_test_saved = df_test_2017["Target"]

y_pred_saved = xgb_tuned_2017.predict(X_test_saved)
y_prob_saved = xgb_tuned_2017.predict_proba(X_test_saved)[:, 1]

missing_mask = X_test_saved[["Flow Bytes/s", "Flow Packets/s"]].isna().any(axis=1)

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob)
    }

missing_results = evaluate_predictions(
    y_test_saved[missing_mask],
    y_pred_saved[missing_mask],
    y_prob_saved[missing_mask]
)

complete_results = evaluate_predictions(
    y_test_saved[~missing_mask],
    y_pred_saved[~missing_mask],
    y_prob_saved[~missing_mask]
)

print("TEST SPLIT FILE:", test_idx_path)
print("TEST ROWS:", len(df_test_2017))
print("ROWS WITH MISSING:", missing_mask.sum())
print()

print("ROWS WITH MISSING VALUES")
print(missing_results)
print(confusion_matrix(
    y_test_saved[missing_mask],
    y_pred_saved[missing_mask]
))
print()

print("ROWS WITHOUT MISSING VALUES")
print(complete_results)
print(confusion_matrix(
    y_test_saved[~missing_mask],
    y_pred_saved[~missing_mask]
))

TEST SPLIT FILE: ..\outputs\splits\retained_test_idx.npy
TEST ROWS: 566149
ROWS WITH MISSING: 451

ROWS WITH MISSING VALUES
{'Accuracy': 0.9977827050997783, 'Precision': 1.0, 'Recall': 0.6666666666666666, 'F1': 0.8, 'ROC-AUC': 0.9977678571428572, 'PR-AUC': 0.8333333333333333}
[[448   0]
 [  1   2]]

ROWS WITHOUT MISSING VALUES
{'Accuracy': 0.9990295175164133, 'Precision': 0.9965902989081797, 'Recall': 0.9984936383117989, 'F1': 0.9975410607031043, 'ROC-AUC': 0.999972655058037, 'PR-AUC': 0.9998777120564011}
[[453790    381]
 [   168 111359]]
